# LeetCode 1325 — Delete Leaves With a Given Value bằng Python

Link đề: https://leetcode.com/problems/delete-leaves-with-a-given-value/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def removeLeafNodes(self, root: TreeNode | None, target: int) -> TreeNode | None:
        ...
```

## 1. Mục tiêu học

1. Tự dựng `TreeNode`, hàm dựng cây từ mảng level-order (có `None`) và hàm chuyển cây ngược về mảng để kiểm thử.
2. Hiểu vì sao bài này cần **duyệt hậu thứ tự (post-order)**: xử lý hai con **trước**, rồi mới quyết định số phận của cha.
3. Nắm kỹ thuật "**hàm đệ quy trả về cây con mới**": `node.left = dfs(node.left)` — trả về `None` nghĩa là "xoá".
4. Viết được 4 cách: lặp xoá nhiều vòng (baseline) → DFS hậu thứ tự đệ quy → hậu thứ tự bằng ngăn xếp (không đệ quy) → "bóc lá" bằng hàng đợi + con trỏ cha.
5. Tránh lỗi: kiểm tra lá **trước** khi xử lý con, gọi đệ quy mà **quên gán lại**, quên trường hợp **chính gốc** bị xoá, đệ quy quá sâu với cây lệch.

## 2. Đề bài & ví dụ

Cho cây nhị phân `root` và số nguyên `target`. Xoá **mọi nút lá** có giá trị `target`. Sau khi xoá, nếu **cha** của nó trở thành lá và cũng có giá trị `target` thì cha cũng bị xoá; cứ tiếp tục như vậy **cho đến khi không xoá được nữa**. Trả về gốc của cây kết quả (có thể là `None`).

**Ví dụ 1:** `root = [1,2,3,2,null,2,4], target = 2` → `[1,null,3,null,4]`

**Ví dụ 2:** `root = [1,3,3,3,2], target = 3` → `[1,3,null,null,2]`

**Ví dụ 3:** `root = [1,2,null,2,null,2], target = 2` → `[1]` (xoá dây chuyền từng tầng).

**Ràng buộc:** số nút trong `[1, 3000]`; `1 <= Node.val, target <= 1000`.

Mảng là biểu diễn **level-order** kiểu LeetCode: `None` (null) đánh dấu con vắng mặt, các `None` cuối mảng bị bỏ.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i])
            q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i])
            q.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    """Chuyển cây về mảng level-order kiểu LeetCode (bỏ None ở cuối)."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

examples = [
    ([1, 2, 3, 2, None, 2, 4], 2, [1, None, 3, None, 4]),
    ([1, 3, 3, 3, 2], 3, [1, 3, None, None, 2]),
    ([1, 2, None, 2, None, 2], 2, [1]),
]
for arr, t, want in examples:
    print(arr, "target =", t, "-> mong đợi", want, "| dựng-lại:", tree_to_list(build_tree(arr)))

[1, 2, 3, 2, None, 2, 4] target = 2 -> mong đợi [1, None, 3, None, 4] | dựng-lại: [1, 2, 3, 2, None, 2, 4]
[1, 3, 3, 3, 2] target = 3 -> mong đợi [1, 3, None, None, 2] | dựng-lại: [1, 3, 3, 3, 2]
[1, 2, None, 2, None, 2] target = 2 -> mong đợi [1] | dựng-lại: [1, 2, None, 2, None, 2]


## 3. Kiến thức nền

### 3.1 Lớp `TreeNode` và cây nhị phân

Mỗi nút có `val`, `left`, `right`. **Lá** là nút không có con nào: `node.left is None and node.right is None`. Ta viết hàm vẽ cây dạng thụt lề để dễ quan sát.

In [2]:
def is_leaf(node):
    return node is not None and node.left is None and node.right is None

def show(root, indent=0, tag="root"):
    if root is None:
        return
    print("    " * indent + f"{tag}: {root.val}" + ("  (lá)" if is_leaf(root) else ""))
    show(root.left, indent + 1, "L")
    show(root.right, indent + 1, "R")

show(build_tree(examples[0][0]))

root: 1
    L: 2
        L: 2  (lá)
    R: 3
        L: 2  (lá)
        R: 4  (lá)


### 3.2 Level-order ↔ cây

`build_tree` dùng hàng đợi: lấy lần lượt từng nút, gán hai phần tử kế tiếp của mảng làm con trái/phải. `tree_to_list` làm ngược lại: BFS, in `None` cho chỗ trống rồi cắt `None` thừa ở cuối. Hai hàm phải **khớp nhau**: `tree_to_list(build_tree(a)) == a` với mảng chuẩn LeetCode.

In [3]:
for a in [[1, None, 3, None, 4], [1, 3, None, None, 2], [5], []]:
    print(a, "->", tree_to_list(build_tree(a)))

[1, None, 3, None, 4] -> [1, None, 3, None, 4]
[1, 3, None, None, 2] -> [1, 3, None, None, 2]
[5] -> [5]
[] -> []


### 3.3 Ba kiểu duyệt DFS

- **Tiền thứ tự (pre-order):** gốc → trái → phải.
- **Trung thứ tự (in-order):** trái → gốc → phải.
- **Hậu thứ tự (post-order):** trái → phải → gốc. Khi xử lý một nút, **hai cây con đã xong** ⇒ hợp với bài "quyết định của cha phụ thuộc kết quả của con".

In [4]:
def preorder(r):
    return [r.val] + preorder(r.left) + preorder(r.right) if r else []

def postorder(r):
    return postorder(r.left) + postorder(r.right) + [r.val] if r else []

t = build_tree([1, 2, 3, 4, 5])
print("pre :", preorder(t))
print("post:", postorder(t))

pre : [1, 2, 4, 5, 3]
post: [4, 5, 2, 3, 1]


### 3.4 Hàm đệ quy "trả về cây con mới"

Mẫu rất hay gặp khi **sửa cấu trúc cây**: hàm nhận gốc của một cây con và trả về gốc **mới** của cây con đó (có thể là `None`). Cha gán lại: `node.left = f(node.left)`. Nhờ vậy không cần biết "cha là ai" khi xoá.

In [5]:
def remove_all_value(node, v):
    """Ví dụ nhỏ: xoá mọi LÁ có giá trị v (chỉ một lượt, không dây chuyền)."""
    if node is None:
        return None
    if is_leaf(node) and node.val == v:
        return None
    node.left = remove_all_value(node.left, v)
    node.right = remove_all_value(node.right, v)
    return node

print(tree_to_list(remove_all_value(build_tree([1, 2, 3, 2, None, 2, 4]), 2)))

[1, 2, 3, None, None, None, 4]


### 3.5 Giới hạn đệ quy của Python

Python mặc định cho khoảng 1000 tầng đệ quy. Cây lệch 3000 nút (giống danh sách liên kết) sẽ làm đệ quy sâu 3000 tầng ⇒ `RecursionError` khi chạy local. LeetCode nâng giới hạn này nên lời giải đệ quy vẫn qua, nhưng ta nên biết cách viết **không đệ quy** (Cách 3, 4).

In [6]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def depth(r):
    return 1 + max(depth(r.left), depth(r.right)) if r else 0

small_skew = build_tree([1, 2, None, 3, None, 4, None, 5])   # lệch trái 5 nút
print("độ sâu cây lệch 5 nút:", depth(small_skew), "-> cây lệch n nút cần n tầng đệ quy")

giới hạn đệ quy hiện tại: 1000
độ sâu cây lệch 5 nút: 5 -> cây lệch n nút cần n tầng đệ quy


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: `[1,2,null,2,null,2]`, target `2` — cây là một dây chuyền lệch trái:

```
1
└─L 2
   └─L 2
      └─L 2   (lá, = 2)
```

- Vòng 1: lá duy nhất là `2` ở tầng sâu nhất ⇒ xoá. Cha của nó (`2`) giờ thành lá.
- Vòng 2: lá `2` (tầng 3) ⇒ xoá. Cha (`2` tầng 2) thành lá.
- Vòng 3: xoá tiếp. Còn lại `1` — là lá nhưng `1 ≠ 2` ⇒ dừng. Kết quả `[1]` ✔.

**Quan sát:** việc xoá **lan từ dưới lên**. Nếu ta xử lý con **trước** cha (hậu thứ tự), thì khi tới cha, ta đã biết các con còn hay mất ⇒ quyết định xoá cha **ngay trong một lượt**, không cần lặp nhiều vòng.

In [7]:
arr, t, want = examples[2]
root = build_tree(arr)
rnd = 0
while True:
    before = tree_to_list(root)
    root = remove_all_value(root, t)
    after = tree_to_list(root)
    if before == after:
        break
    rnd += 1
    print(f"vòng {rnd}: {before} -> {after}")
print("kết quả:", tree_to_list(root), "| mong đợi", want)

vòng 1: [1, 2, None, 2, None, 2] -> [1, 2, None, 2]
vòng 2: [1, 2, None, 2] -> [1, 2]
vòng 3: [1, 2] -> [1]
kết quả: [1] | mong đợi [1]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** mô phỏng đúng như đề: lặp lại "xoá mọi lá hiện tại có giá trị `target`" cho đến khi một vòng không xoá được gì. Mỗi vòng là một lượt duyệt cây; ta đếm số nút bị xoá để biết khi nào dừng.

Lưu ý: trong một vòng, ta xác định "lá" theo **trạng thái đầu vòng** — dùng tiền thứ tự (kiểm tra nút trước, rồi mới đi xuống con) nên một nút chỉ bị xoá nếu nó **đã là lá** khi ta tới nó.

In [8]:
def remove_leaf_nodes_brute(root, target):
    def one_round(node):
        """Trả về (cây mới, số nút đã xoá) — chỉ xoá lá hiện tại."""
        if node is None:
            return None, 0
        if node.left is None and node.right is None:
            return (None, 1) if node.val == target else (node, 0)
        node.left, a = one_round(node.left)
        node.right, b = one_round(node.right)
        return node, a + b

    while True:
        root, removed = one_round(root)
        if removed == 0 or root is None:
            return root

for arr, t, want in examples:
    got = tree_to_list(remove_leaf_nodes_brute(build_tree(arr), t))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[1, None, 3, None, 4] | mong đợi [1, None, 3, None, 4] | OK
[1, 3, None, None, 2] | mong đợi [1, 3, None, None, 2] | OK
[1] | mong đợi [1] | OK


**Độ phức tạp:** mỗi vòng `O(n)`; số vòng có thể bằng chiều cao `h` (dây chuyền toàn `target`) ⇒ `O(n·h)`, xấu nhất `O(n²)` với cây lệch (`n = 3000` ⇒ ~9·10^6 bước — vẫn chạy được nhưng lãng phí). Bộ nhớ `O(h)` cho đệ quy.

**Nút thắt:** mỗi vòng duyệt lại **toàn bộ** cây chỉ để xoá thêm một tầng. Nếu duyệt **từ dưới lên** thì mọi thông tin "con đã bị xoá" có sẵn khi xét cha ⇒ chỉ cần **một** lượt.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "nút `x` cuối cùng bị xoá" ⇔ `x.val == target` **và** mọi con của `x` (nếu có) đều cuối cùng bị xoá. Đây là định nghĩa **đệ quy từ dưới lên** ⇒ duyệt **hậu thứ tự**.

**Hàm `dfs(node)`** trả về gốc mới của cây con:
1. Nếu `node is None` ⇒ trả `None`.
2. `node.left = dfs(node.left)`; `node.right = dfs(node.right)` (xử lý con **trước**).
3. **Nếu** giờ `node` là lá (cả hai con đều `None`) **và** `node.val == target` **thì** trả `None` (xoá), **ngược lại** trả `node`.

**Invariant:** khi `dfs(node)` trả về, cây con kết quả **không còn lá nào bằng `target`**, và mọi nút còn lại đều phải còn (không xoá thừa). Gốc cả cây cũng đi qua bước 3 ⇒ có thể trả về `None`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS hậu thứ tự đệ quy (lời giải chuẩn)

**Từng đoạn:** gọi đệ quy cho con trái, gán lại; gọi cho con phải, gán lại; cuối cùng kiểm tra chính nút này. Thứ tự này là chìa khoá: nếu kiểm tra trước khi xử lý con, ta sẽ bỏ sót hiệu ứng dây chuyền.

In [9]:
def remove_leaf_nodes_dfs(root, target):
    if root is None:
        return None
    root.left = remove_leaf_nodes_dfs(root.left, target)
    root.right = remove_leaf_nodes_dfs(root.right, target)
    if root.left is None and root.right is None and root.val == target:
        return None
    return root

for arr, t, want in examples:
    got = tree_to_list(remove_leaf_nodes_dfs(build_tree(arr), t))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[1, None, 3, None, 4] | mong đợi [1, None, 3, None, 4] | OK
[1, 3, None, None, 2] | mong đợi [1, 3, None, None, 2] | OK
[1] | mong đợi [1] | OK


Code nộp LeetCode (Cách 2):

In [10]:
class SolutionDFS:
    def removeLeafNodes(self, root: Optional[TreeNode], target: int) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left = self.removeLeafNodes(root.left, target)
        root.right = self.removeLeafNodes(root.right, target)
        if root.left is None and root.right is None and root.val == target:
            return None
        return root

print([tree_to_list(SolutionDFS().removeLeafNodes(build_tree(a), t)) for a, t, _ in examples])

[[1, None, 3, None, 4], [1, 3, None, None, 2], [1]]


**Độ phức tạp:** mỗi nút được thăm đúng một lần ⇒ `O(n)` thời gian; `O(h)` bộ nhớ ngăn xếp đệ quy (`h` = chiều cao, xấu nhất `n`).

### 7.2 Cách 3 — Hậu thứ tự bằng ngăn xếp (không đệ quy)

**Ý tưởng:** mô phỏng ngăn xếp đệ quy. Mỗi phần tử ngăn xếp là `(node, parent, is_left, visited)`:
- lần đầu gặp (`visited = False`): đẩy lại chính nó với `visited = True`, rồi đẩy hai con ⇒ hai con được xử lý **trước**;
- lần thứ hai (`visited = True`): hai con đã xong ⇒ nếu nút là lá và bằng `target` thì **cắt** khỏi cha (`parent.left = None` hoặc `parent.right = None`).

Dùng một **nút giả** `dummy` làm cha của gốc để xử lý đồng nhất trường hợp gốc bị xoá.

In [11]:
def remove_leaf_nodes_stack(root, target):
    dummy = TreeNode(0, root)
    stack = [(root, dummy, True, False)]
    while stack:
        node, parent, is_left, visited = stack.pop()
        if node is None:
            continue
        if not visited:
            stack.append((node, parent, is_left, True))
            stack.append((node.right, node, False, False))
            stack.append((node.left, node, True, False))
        elif node.left is None and node.right is None and node.val == target:
            if is_left:
                parent.left = None
            else:
                parent.right = None
    return dummy.left

for arr, t, want in examples:
    got = tree_to_list(remove_leaf_nodes_stack(build_tree(arr), t))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[1, None, 3, None, 4] | mong đợi [1, None, 3, None, 4] | OK
[1, 3, None, None, 2] | mong đợi [1, 3, None, None, 2] | OK
[1] | mong đợi [1] | OK


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionStack:
    def removeLeafNodes(self, root: Optional[TreeNode], target: int) -> Optional[TreeNode]:
        dummy = TreeNode(0, root)
        stack = [(root, dummy, True, False)]
        while stack:
            node, parent, is_left, visited = stack.pop()
            if node is None:
                continue
            if not visited:
                stack.append((node, parent, is_left, True))
                stack.append((node.right, node, False, False))
                stack.append((node.left, node, True, False))
            elif node.left is None and node.right is None and node.val == target:
                if is_left:
                    parent.left = None
                else:
                    parent.right = None
        return dummy.left

print([tree_to_list(SolutionStack().removeLeafNodes(build_tree(a), t)) for a, t, _ in examples])

[[1, None, 3, None, 4], [1, 3, None, None, 2], [1]]


**Độ phức tạp:** mỗi nút vào ngăn xếp 2 lần ⇒ `O(n)` thời gian; ngăn xếp tối đa `O(n)` phần tử (an toàn với cây lệch 3000 nút, không phụ thuộc giới hạn đệ quy).

### 7.3 Cách 4 — "Bóc lá" bằng hàng đợi + con trỏ cha

**Ý tưởng (giống thuật toán Kahn bóc từ lá vào):**
1. BFS một lượt để ghi `parent[node]` và đếm số con `kids[node]`.
2. Đưa mọi **lá có giá trị `target`** vào hàng đợi.
3. Lấy một nút ra, cắt khỏi cha, giảm `kids[cha]`. Nếu cha giờ **hết con** và `cha.val == target` ⇒ đưa cha vào hàng đợi.
4. Nếu gốc bị lấy ra ⇒ trả `None`.

Cách này cho thấy rõ bản chất "**lan truyền từ lá lên gốc**" và mở rộng tốt cho cây tổng quát (nhiều con) hoặc đồ thị dạng cây.

In [13]:
def remove_leaf_nodes_peel(root, target):
    if root is None:
        return None
    parent = {root: None}
    kids = {}
    order = deque([root])
    nodes = []
    while order:
        node = order.popleft()
        nodes.append(node)
        kids[node] = 0
        for ch in (node.left, node.right):
            if ch is not None:
                parent[ch] = node
                kids[node] += 1
                order.append(ch)
    q = deque(nd for nd in nodes if kids[nd] == 0 and nd.val == target)
    while q:
        node = q.popleft()
        p = parent[node]
        if p is None:
            return None                      # chính gốc bị xoá
        if p.left is node:
            p.left = None
        else:
            p.right = None
        kids[p] -= 1
        if kids[p] == 0 and p.val == target:
            q.append(p)
    return root

for arr, t, want in examples:
    got = tree_to_list(remove_leaf_nodes_peel(build_tree(arr), t))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[1, None, 3, None, 4] | mong đợi [1, None, 3, None, 4] | OK
[1, 3, None, None, 2] | mong đợi [1, 3, None, None, 2] | OK
[1] | mong đợi [1] | OK


Code nộp LeetCode (Cách 4):

In [14]:
class SolutionPeel:
    def removeLeafNodes(self, root: Optional[TreeNode], target: int) -> Optional[TreeNode]:
        if root is None:
            return None
        parent = {root: None}
        kids = {}
        bfs = deque([root])
        leaves = deque()
        while bfs:
            node = bfs.popleft()
            kids[node] = 0
            for ch in (node.left, node.right):
                if ch is not None:
                    parent[ch] = node
                    kids[node] += 1
                    bfs.append(ch)
            if kids[node] == 0 and node.val == target:
                leaves.append(node)
        while leaves:
            node = leaves.popleft()
            p = parent[node]
            if p is None:
                return None
            if p.left is node:
                p.left = None
            else:
                p.right = None
            kids[p] -= 1
            if kids[p] == 0 and p.val == target:
                leaves.append(p)
        return root

print([tree_to_list(SolutionPeel().removeLeafNodes(build_tree(a), t)) for a, t, _ in examples])

[[1, None, 3, None, 4], [1, 3, None, None, 2], [1]]


**Độ phức tạp:** `O(n)` thời gian (mỗi nút vào hàng đợi BFS một lần, vào hàng đợi lá tối đa một lần), `O(n)` bộ nhớ cho `parent`, `kids`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** `[1,2,3,2,null,2,4]`, target `2`:

| thứ tự xử lý (hậu thứ tự) | nút | con còn lại sau khi xử lý con | là lá? | val == 2? | trả về |
|---|---|---|---|---|---|
| 1 | 2 (lá, con trái của 2) | — | có | có | `None` |
| 2 | 2 (con trái của 1) | không còn con | có | có | `None` |
| 3 | 2 (con trái của 3) | — | có | có | `None` |
| 4 | 4 | — | có | không | giữ |
| 5 | 3 | chỉ còn phải = 4 | không | — | giữ |
| 6 | 1 | chỉ còn phải = 3 | không | — | giữ |

Kết quả `[1,null,3,null,4]`.

In [15]:
def trace_remove(root, target, depth=0, tag="root"):
    pad = "   " * depth
    if root is None:
        return None
    print(f"{pad}vào {tag}={root.val}")
    root.left = trace_remove(root.left, target, depth + 1, "L")
    root.right = trace_remove(root.right, target, depth + 1, "R")
    leaf = root.left is None and root.right is None
    decision = leaf and root.val == target
    print(f"{pad}ra  {tag}={root.val}: left={'∅' if root.left is None else root.left.val}, "
          f"right={'∅' if root.right is None else root.right.val}, lá={leaf} -> {'XOÁ' if decision else 'giữ'}")
    return None if decision else root

res = trace_remove(build_tree([1, 2, 3, 2, None, 2, 4]), 2)
print("kết quả:", tree_to_list(res))

vào root=1
   vào L=2
      vào L=2
      ra  L=2: left=∅, right=∅, lá=True -> XOÁ
   ra  L=2: left=∅, right=∅, lá=True -> XOÁ
   vào R=3
      vào L=2
      ra  L=2: left=∅, right=∅, lá=True -> XOÁ
      vào R=4
      ra  R=4: left=∅, right=∅, lá=True -> giữ
   ra  R=3: left=∅, right=4, lá=False -> giữ
ra  root=1: left=∅, right=3, lá=False -> giữ
kết quả: [1, None, 3, None, 4]


Trace Cách 4 trên **Ví dụ 3** — xem hàng đợi lá lan dần lên:

In [16]:
def trace_peel(arr, target):
    root = build_tree(arr)
    parent, kids, bfs, nodes = {root: None}, {}, deque([root]), []
    while bfs:
        nd = bfs.popleft()
        nodes.append(nd)
        kids[nd] = sum(ch is not None for ch in (nd.left, nd.right))
        for ch in (nd.left, nd.right):
            if ch is not None:
                parent[ch] = nd
                bfs.append(ch)
    idx = {nd: i for i, nd in enumerate(nodes)}          # đặt tên nút theo thứ tự BFS
    q = deque(nd for nd in nodes if kids[nd] == 0 and nd.val == target)
    print("hàng đợi ban đầu:", [f"#{idx[x]}({x.val})" for x in q])
    while q:
        nd = q.popleft()
        p = parent[nd]
        if p is None:
            print(f"xoá gốc #{idx[nd]} -> cây rỗng")
            return None
        if p.left is nd:
            p.left = None
        else:
            p.right = None
        kids[p] -= 1
        push = kids[p] == 0 and p.val == target
        if push:
            q.append(p)
        print(f"xoá #{idx[nd]}({nd.val}); cha #{idx[p]}({p.val}) còn {kids[p]} con -> {'đẩy cha' if push else 'không đẩy'}")
    print("kết quả:", tree_to_list(root))
    return root

trace_peel([1, 2, None, 2, None, 2], 2)

hàng đợi ban đầu: ['#3(2)']
xoá #3(2); cha #2(2) còn 0 con -> đẩy cha
xoá #2(2); cha #1(2) còn 0 con -> đẩy cha
xoá #1(2); cha #0(1) còn 0 con -> không đẩy
kết quả: [1]


## 9. Code nộp LeetCode

Bản chọn nộp: DFS hậu thứ tự (Cách 2) — ngắn và là lời giải chuẩn. Trên LeetCode lớp `TreeNode` có sẵn.

In [17]:
class Solution:
    def removeLeafNodes(self, root: Optional[TreeNode], target: int) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left = self.removeLeafNodes(root.left, target)
        root.right = self.removeLeafNodes(root.right, target)
        if root.left is None and root.right is None and root.val == target:
            return None
        return root

for arr, t, want in examples:
    got = tree_to_list(Solution().removeLeafNodes(build_tree(arr), t))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

[1, None, 3, None, 4] | mong đợi [1, None, 3, None, 4] | OK
[1, 3, None, None, 2] | mong đợi [1, 3, None, None, 2] | OK
[1] | mong đợi [1] | OK


## 10. Vì sao thuật toán đúng

Định nghĩa: nút `x` **bị xoá cuối cùng** (theo quy trình lặp của đề) khi và chỉ khi `x.val == target` và mọi con của `x` đều bị xoá cuối cùng. (Chiều ⇒: `x` chỉ bị xoá khi là lá, tức các con đã bị xoá trước. Chiều ⇐: khi các con đều đã bị xoá, `x` thành lá có giá trị `target`, quy trình "tiếp tục đến khi không xoá được" sẽ xoá nó.) Thứ tự xoá không ảnh hưởng tập kết quả.

**Quy nạp theo chiều cao cây con:** giả sử `dfs` đúng với hai cây con (trả `None` ⇔ cả cây con bị xoá hết, ngược lại trả cây con đã xoá đúng). Tại `x`, sau khi gán lại, `x.left is None and x.right is None` ⇔ mọi con của `x` đều bị xoá hết ⇒ điều kiện ở bước 3 trùng định nghĩa trên ⇒ `dfs(x)` đúng. Trường hợp cơ sở `None` hiển nhiên.

**Không xoá thừa:** một nút chỉ bị trả `None` khi đúng điều kiện trên; mọi nút có giá trị khác `target`, hoặc còn ít nhất một con, đều được giữ.

## 11. Phân tích Time / Space

Đặt `n` = số nút, `h` = chiều cao cây (`log n ≤ h ≤ n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lặp nhiều vòng | `O(n·h)` (xấu nhất `O(n²)`) | `O(h)` | mô phỏng đề |
| 2. DFS hậu thứ tự đệ quy | `O(n)` | `O(h)` ngăn xếp gọi hàm | chuẩn, ngắn |
| 3. Ngăn xếp tường minh | `O(n)` | `O(n)` | không phụ thuộc giới hạn đệ quy |
| 4. Bóc lá + con trỏ cha | `O(n)` | `O(n)` | dễ mở rộng cho cây nhiều con |

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Lưu ý: các lời giải **sửa cây tại chỗ**, nên mỗi lời giải phải chạy trên một cây **dựng mới**.

In [18]:
sys.setrecursionlimit(10000)      # cho phép test cây lệch 3000 nút với các cách đệ quy

solvers = {
    "brute": remove_leaf_nodes_brute,
    "dfs": remove_leaf_nodes_dfs,
    "stack": remove_leaf_nodes_stack,
    "peel": remove_leaf_nodes_peel,
    "submit": lambda r, t: Solution().removeLeafNodes(r, t),
    "submit_stack": lambda r, t: SolutionStack().removeLeafNodes(r, t),
    "submit_peel": lambda r, t: SolutionPeel().removeLeafNodes(r, t),
}

def run_all(arr, t):
    return {name: tree_to_list(f(build_tree(arr), t)) for name, f in solvers.items()}

cases = examples + [
    ([2], 2, []),                          # một nút = target -> rỗng
    ([1], 2, [1]),                         # một nút khác target
    ([2, 2, 2], 2, []),                    # cả cây bị xoá
    ([1, 1, 1], 1, []),
    ([2, 1, 2], 2, [2, 1]),                # gốc = target nhưng còn con
    ([1, 2, 2, 2, 2, 2, 2], 2, [1]),       # cây đầy đủ toàn 2 dưới gốc
    ([3, 3, None, 3, 4], 3, [3, 3, None, None, 4]),
]
for arr, t, want in cases:
    res = run_all(arr, t)
    for name, got in res.items():
        assert got == want, (name, arr, t, got, want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 10 case qua cho 7 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh cây ngẫu nhiên với giá trị trong tập nhỏ `{1, 2, 3}` (để `target` xuất hiện nhiều, tạo nhiều dây chuyền).

In [19]:
def random_tree_arr(rng, n, vals=(1, 2, 3)):
    """Sinh mảng level-order của một cây ngẫu nhiên đúng n nút."""
    root = TreeNode(rng.choice(vals))
    nodes = [root]
    while len(nodes) < n:
        p = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(p, side) is None:
            ch = TreeNode(rng.choice(vals))
            setattr(p, side, ch)
            nodes.append(ch)
    return tree_to_list(root)

rng = random.Random(1325)
for _ in range(800):
    arr = random_tree_arr(rng, rng.randint(1, 25))
    t = rng.choice([1, 2, 3])
    res = run_all(arr, t)
    want = res["brute"]
    assert all(v == want for v in res.values()), (arr, t, res)
print("800 cây ngẫu nhiên: mọi cách khớp brute force")

800 cây ngẫu nhiên: mọi cách khớp brute force


### 12.3 Cây lớn: 3000 nút, cây lệch toàn `target`

In [20]:
n = 3000
skew = [2] + [2, None] * (n - 1)        # lệch trái: mỗi nút chỉ có con trái
skew_arr = tree_to_list(build_tree(skew))
print("số nút:", sum(v is not None for v in skew_arr))
for name in ["dfs", "stack", "peel", "submit"]:
    t0 = time.time()
    got = tree_to_list(solvers[name](build_tree(skew_arr), 2))
    print(f"{name:7}: {got}  ({time.time() - t0:.3f}s)")
t0 = time.time()
got = tree_to_list(remove_leaf_nodes_brute(build_tree(skew_arr), 2))
print(f"brute  : {got}  ({time.time() - t0:.3f}s)  <- O(n^2)")
rng = random.Random(3)
big = random_tree_arr(rng, 3000)
assert len(set(map(str, run_all(big, 2).values()))) == 1
print("cây ngẫu nhiên 3000 nút: mọi cách trùng nhau")

số nút: 3000
dfs    : []  (0.002s)
stack  : []  (0.004s)
peel   : []  (0.004s)
submit : []  (0.003s)


brute  : []  (1.098s)  <- O(n^2)
cây ngẫu nhiên 3000 nút: mọi cách trùng nhau


## 13. Các lỗi người mới hay gặp

Để thấy lỗi giới hạn đệ quy, ta tạm hạ giới hạn về mặc định ở Lỗi 4 rồi khôi phục.

### Lỗi 1 — Kiểm tra lá trước khi xử lý con (tiền thứ tự)

In [21]:
def wrong_preorder(root, target):
    if root is None:
        return None
    if root.left is None and root.right is None and root.val == target:
        return None
    root.left = wrong_preorder(root.left, target)
    root.right = wrong_preorder(root.right, target)
    return root

arr, t, want = examples[2]
print("sai :", tree_to_list(wrong_preorder(build_tree(arr), t)))
print("đúng:", tree_to_list(Solution().removeLeafNodes(build_tree(arr), t)))

sai : [1, 2, None, 2]
đúng: [1]


**Vì sao sai:** khi xét nút `2` ở tầng giữa, nó **chưa** là lá (con của nó chưa bị xoá) ⇒ bị giữ lại; sau đó con bị xoá nhưng ta không quay lại xét cha nữa ⇒ chỉ xoá được một tầng. **Sửa:** xử lý hai con **trước**, kiểm tra nút **sau** (hậu thứ tự).

### Lỗi 2 — Gọi đệ quy nhưng quên gán lại con

In [22]:
def wrong_no_assign(root, target):
    if root is None:
        return None
    wrong_no_assign(root.left, target)        # SAI: kết quả bị bỏ đi
    wrong_no_assign(root.right, target)
    if root.left is None and root.right is None and root.val == target:
        return None
    return root

arr, t, want = examples[0]
print("sai :", tree_to_list(wrong_no_assign(build_tree(arr), t)))
print("đúng:", tree_to_list(Solution().removeLeafNodes(build_tree(arr), t)))

sai : [1, 2, 3, 2, None, 2, 4]
đúng: [1, None, 3, None, 4]


**Vì sao sai:** trả `None` chỉ là **giá trị trả về**, không tự cắt liên kết `root.left`. Nếu không gán lại, cây không đổi gì. **Sửa:** `root.left = dfs(root.left)`, `root.right = dfs(root.right)`.

### Lỗi 3 — Chỉ xử lý các con, quên rằng gốc cũng có thể bị xoá

In [23]:
def wrong_keep_root(root, target):
    def dfs(node):
        if node is None:
            return None
        node.left = dfs(node.left)
        node.right = dfs(node.right)
        if node.left is None and node.right is None and node.val == target:
            return None
        return node
    root.left = dfs(root.left)         # SAI: chỉ xử lý con của gốc
    root.right = dfs(root.right)
    return root

print("sai :", tree_to_list(wrong_keep_root(build_tree([2, 2, 2]), 2)))
print("đúng:", tree_to_list(Solution().removeLeafNodes(build_tree([2, 2, 2]), 2)))

sai : [2]
đúng: []


**Vì sao sai:** với `[2,2,2]`, target `2`, sau khi xoá hai lá, gốc `2` thành lá và cũng phải bị xoá ⇒ đáp án là cây rỗng `[]`. Code sai trả `[2]`. **Sửa:** gọi `return dfs(root)` (cho gốc đi qua cùng một luật), hoặc dùng nút giả `dummy` như Cách 3.

### Lỗi 4 — Đệ quy quá sâu khi chạy local với cây lệch

In [24]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    Solution().removeLeafNodes(build_tree(skew_arr), 2)
    print("không lỗi")
except RecursionError as e:
    print("đệ quy văng lỗi:", type(e).__name__, "với cây lệch", n, "nút")
finally:
    sys.setrecursionlimit(old)
print("ngăn xếp tường minh vẫn chạy:", tree_to_list(remove_leaf_nodes_stack(build_tree(skew_arr), 2)))

đệ quy văng lỗi: RecursionError với cây lệch 3000 nút
ngăn xếp tường minh vẫn chạy: []


**Vì sao sai:** cây lệch 3000 nút ⇒ 3000 tầng đệ quy > giới hạn mặc định 1000 của Python. Trên LeetCode thường vẫn qua (giới hạn được nâng), nhưng trong phỏng vấn/dự án nên biết. **Sửa:** `sys.setrecursionlimit` đủ lớn (khi chắc stack C chịu được), hoặc dùng Cách 3/Cách 4.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xoá/cắt nút, rồi cha có thể bị ảnh hưởng, **tiếp tục cho đến khi**…" ⇒ **hậu thứ tự** (thông tin đi từ con lên cha).
- Sửa cấu trúc cây ⇒ mẫu `node.left = f(node.left)`; trả `None` để xoá.
- Gốc có thể bị xoá ⇒ luôn trả về gốc mới (`return f(root)`), hoặc dùng **nút giả**.
- Lời giải "lặp đến khi không đổi" thường có thể thay bằng **một lượt** theo đúng thứ tự phụ thuộc (ở đây: dưới lên).
- Test bài cây: viết sẵn `build_tree` / `tree_to_list` để so sánh bằng mảng.

## 15. Biến thể của bài

1. **LeetCode 366 — Find Leaves of Binary Tree** (Premium): lặp "bóc tất cả lá" và gom từng đợt — chính là **chiều cao từ dưới lên** của mỗi nút (Bài tập 5).
2. **LeetCode 814 — Binary Tree Pruning:** xoá mọi cây con không chứa số `1` — cùng mẫu hậu thứ tự trả `None`.
3. **LeetCode 1110 — Delete Nodes And Return Forest:** xoá các nút trong một tập, trả về rừng.
4. Xoá lá có giá trị thuộc **một tập** `targets` thay vì một số.
5. Cây **nhiều con** (N-ary): thay hai con bằng danh sách con, lọc những con trả về `None`.

In [25]:
def prune_tree(root):
    """LeetCode 814: xoá mọi cây con không chứa số 1."""
    if root is None:
        return None
    root.left = prune_tree(root.left)
    root.right = prune_tree(root.right)
    if root.left is None and root.right is None and root.val == 0:
        return None
    return root

def remove_leaves_in_set(root, targets):
    if root is None:
        return None
    root.left = remove_leaves_in_set(root.left, targets)
    root.right = remove_leaves_in_set(root.right, targets)
    return None if root.left is None and root.right is None and root.val in targets else root

print("814:", tree_to_list(prune_tree(build_tree([1, None, 0, 0, 1]))))
print("tập target:", tree_to_list(remove_leaves_in_set(build_tree([1, 2, 3, 4, 5, 6, 7]), {2, 4, 5, 7})))

814: [1, None, 0, None, 1]
tập target: [1, None, 3, 6]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lặp nhiều vòng | mỗi vòng xoá lá hiện tại, đến khi không đổi | `O(n·h)` | `O(h)` | kiểm chứng, đúng nguyên văn đề |
| 2. DFS hậu thứ tự | con trước, cha sau, trả `None` để xoá | `O(n)` | `O(h)` | mặc định, ngắn nhất |
| 3. Ngăn xếp tường minh | mô phỏng hậu thứ tự, nút giả `dummy` | `O(n)` | `O(n)` | cây rất sâu, tránh đệ quy |
| 4. Bóc lá + cha | hàng đợi lá, lan dần lên cha | `O(n)` | `O(n)` | cây nhiều con, cần thứ tự xoá |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`removeLeafNodes([1,1,1,1,None,None,1], target=1)` trả về mảng nào?

### Bài 2 (tính tay)
Tính tay `removeLeafNodes([5,5,6,5,5,None,5], target=5)`. Nút nào bị xoá theo thứ tự hậu thứ tự?

### Bài 3 (điền TODO)
Hoàn thiện `remove_todo` (hậu thứ tự, đệ quy).

### Bài 4 (tự viết lại)
Viết `count_removed(root, target)` trả về **số nút** bị xoá (không cần trả cây). Gợi ý: hàm phụ trả `(cây mới, số nút xoá)`.

### Bài 5 (mini challenge — LeetCode 366)
Viết `find_leaves(root)` trả về danh sách các đợt lá: đợt 1 là mọi lá ban đầu, xoá đi; đợt 2 là các lá mới; ... Ví dụ `[1,2,3,4,5]` → `[[4,5,3],[2],[1]]`. Làm trong **một lượt** DFS.

In [26]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def remove_todo(root, target):
    if root is None:
        return None
    # TODO: xử lý hai con và gán lại
    # TODO: nếu root là lá và bằng target thì trả None
    return root

def count_removed_todo(root, target):
    # TODO
    pass

def find_leaves_todo(root):
    # TODO: chiều cao từ dưới lên = chỉ số đợt
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Vẽ cây: gốc 1, con trái 1 (có con trái 1), con phải 1 (có con phải 1). (2) Mọi nút đều bằng target. (3) Xoá lan đến gốc ⇒ `[]`.

**Bài 2:** (1) Cây: `5 → (5 → 5, 5), (6 → ∅, 5)`. (2) Hai lá `5` dưới con trái bị xoá, con trái `5` thành lá ⇒ xoá. (3) Lá `5` dưới `6` bị xoá; `6 ≠ 5` dừng; gốc còn con `6` ⇒ `[5,None,6]`.

**Bài 3:** (1) `root.left = remove_todo(root.left, target)`. (2) Tương tự cho phải. (3) Điều kiện xoá: hai con `None` và `root.val == target`.

**Bài 4:** (1) Hàm phụ trả cặp. (2) Cộng số xoá của hai con. (3) Nếu nút hiện tại bị xoá thì cộng thêm 1.

**Bài 5:** (1) Định nghĩa `h(None) = -1`, `h(lá) = 0`, `h(x) = 1 + max(h(trái), h(phải))`. (2) Nút có `h = k` bị bóc ở đợt `k`. (3) Duyệt hậu thứ tự, `res[h].append(val)`, tạo thêm danh sách khi `h == len(res)`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
b1 = tree_to_list(Solution().removeLeafNodes(build_tree([1, 1, 1, 1, None, None, 1]), 1))
b2 = tree_to_list(Solution().removeLeafNodes(build_tree([5, 5, 6, 5, 5, None, 5]), 5))
print("Bài 1:", b1, "| Bài 2:", b2)
assert b1 == [] and b2 == [5, None, 6]

# Bài 3
def remove_sol(root, target):
    if root is None:
        return None
    root.left = remove_sol(root.left, target)
    root.right = remove_sol(root.right, target)
    if root.left is None and root.right is None and root.val == target:
        return None
    return root

for arr, t, want in cases:
    assert tree_to_list(remove_sol(build_tree(arr), t)) == want

# Bài 4
def count_removed(root, target):
    def go(node):
        if node is None:
            return None, 0
        node.left, a = go(node.left)
        node.right, b = go(node.right)
        if node.left is None and node.right is None and node.val == target:
            return None, a + b + 1
        return node, a + b
    return go(root)[1]

def count_nodes(arr):
    return sum(v is not None for v in arr)

rng = random.Random(44)
for _ in range(300):
    arr = random_tree_arr(rng, rng.randint(1, 20))
    t = rng.choice([1, 2, 3])
    after = tree_to_list(Solution().removeLeafNodes(build_tree(arr), t))
    assert count_removed(build_tree(arr), t) == count_nodes(arr) - count_nodes(after)
print("Bài 1–4: qua")

Bài 1: [] | Bài 2: [5, None, 6]
Bài 1–4: qua


In [28]:
# Bài 5
def find_leaves(root):
    res = []

    def h(node):
        if node is None:
            return -1
        k = 1 + max(h(node.left), h(node.right))
        if k == len(res):
            res.append([])
        res[k].append(node.val)
        return k

    h(root)
    return res

def find_leaves_brute(root):
    """Kiểm chứng: lặp bóc lá thật sự (dựa trên Cách 1 với 'mọi giá trị')."""
    res = []
    while root is not None:
        layer = []

        def peel(node):
            if node is None:
                return None
            if node.left is None and node.right is None:
                layer.append(node.val)
                return None
            node.left = peel(node.left)
            node.right = peel(node.right)
            return node

        root = peel(root)
        res.append(layer)
    return res

print("Bài 5:", find_leaves(build_tree([1, 2, 3, 4, 5])))
assert find_leaves(build_tree([1, 2, 3, 4, 5])) == [[4, 5, 3], [2], [1]]
rng = random.Random(366)
for _ in range(300):
    arr = random_tree_arr(rng, rng.randint(1, 20), vals=tuple(range(1, 50)))
    assert find_leaves(build_tree(arr)) == find_leaves_brute(build_tree(arr))
print("Bài 5: 300 cây ngẫu nhiên khớp cách bóc lá lặp")

Bài 5: [[4, 5, 3], [2], [1]]
Bài 5: 300 cây ngẫu nhiên khớp cách bóc lá lặp


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template hậu thứ tự "trả về cây con mới":**

```python
def dfs(node):
    if node is None:
        return None
    node.left = dfs(node.left)      # con trước
    node.right = dfs(node.right)
    if <node cần bị xoá dựa trên trạng thái mới của con>:
        return None
    return node
return dfs(root)                    # gốc cũng đi qua cùng luật
```

**Checklist trước khi submit:**
- [ ] Xử lý **hai con trước**, kiểm tra nút **sau**.
- [ ] Có **gán lại** `root.left = ...`, `root.right = ...`.
- [ ] Trả về kết quả của chính gốc (có thể `None`).
- [ ] Điều kiện lá: `left is None and right is None` (kiểm tra **sau** khi gán lại).
- [ ] Cây sâu: cân nhắc phiên bản ngăn xếp.
- [ ] Đúng chữ ký: `removeLeafNodes(self, root, target) -> Optional[TreeNode]`.